In [2]:
!pip install -U "mineru[core,vllm]"

In [11]:
!pip uninstall -y vllm
!pip install vllm==0.10.2

Found existing installation: vllm 0.20.2
Uninstalling vllm-0.20.2:
  Successfully uninstalled vllm-0.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 59.3 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 107.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 99.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 18.9 MB/s  0:00:22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 29.8 MB/s  0:00:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 31.1 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 122.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 106.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 65.5 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 68.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 129.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.

In [5]:
%env LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:/usr/lib64-nvidia

env: LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:/usr/lib64-nvidia


In [1]:
!mineru -p /content/handwritten_en_0000.jpg -o ./output -b vlm-engine

2026-06-24 13:55:22.207 | INFO     | mineru.cli.client:run_orchestrated_cli:953 - Started local mineru-api at http://127.0.0.1:55935
2026-06-24 13:55:25.945 | INFO     | __main__:create_app:236 - Request concurrency limited to 3
Start MinerU FastAPI Service: http://127.0.0.1:55935
API documentation: http://127.0.0.1:55935/docs
INFO:     Started server process [12014]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:55935 (Press CTRL+C to quit)
2026-06-24 13:55:26.236 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 1/1 | 1 document, 1 page in this batch | 1 page total | task#1 [handwritten_en_0000]
2026-06-24 13:55:27.535 | INFO     | mineru.utils.engine_utils:get_vlm_engine:34 - Using vllm-async-engine as the inference engine for VLM.
2026-06-24 13:55:28.679920: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions i

In [32]:
!nohup mineru-api \
    --enable-vlm-preload true \
    > mineru_api.log 2>&1 &

# !mineru-api --enable-vlm-preload true
# !mineru --api-url http://127.0.0.1:8000 -p /content/handwritten_en_0000.jpg -o ./output -b vlm-engine

In [42]:
import os

zip_file_path = "/content/table_by_level.zip"
output_dir = "documents"

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Unzip the file
!unzip -o "{zip_file_path}" -d "{output_dir}"

print(f"Unzipped {zip_file_path} to {output_dir}")

Archive:  /content/table_by_level.zip
   creating: documents/level_1/
  inflating: documents/level_1/table_0016.jpg  
  inflating: documents/level_1/table_0096.jpg  
  inflating: documents/level_1/table_0101.jpg  
  inflating: documents/level_1/table_0135.jpg  
  inflating: documents/level_1/table_0138.jpg  
   creating: documents/level_2/
  inflating: documents/level_2/table_0079.jpg  
  inflating: documents/level_2/table_0155.jpg  
  inflating: documents/level_2/table_0203.jpg  
  inflating: documents/level_2/table_0281.jpg  
  inflating: documents/level_2/table_0704.jpg  
   creating: documents/level_3/
  inflating: documents/level_3/table_0143.jpg  
  inflating: documents/level_3/table_0331.jpg  
  inflating: documents/level_3/table_0534.jpg  
  inflating: documents/level_3/table_0584.jpg  
  inflating: documents/level_3/table_0804.jpg  
   creating: documents/level_4/
  inflating: documents/level_4/table_0047.jpg  
  inflating: documents/level_4/table_0329.jpg  
  inflating: docum

In [36]:
import os
import time
import json
import psutil
import requests
import subprocess
import threading

from datetime import datetime

# --- Global Config ---
INPUT_DIR = "/content/documents"
OUTPUT_DIR = "output"

MINERU_API_URL = "http://127.0.0.1:8000"

CSV_LOG_NAME = "log.csv"
JSON_LOG_NAME = "log.json"

# --- GPU MONITOR ---
def get_gpu_stats():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,nounits,noheader"]
        ).decode("utf-8")

        util, mem_used, mem_total = result.strip().split(", ")
        return float(util), float(mem_used), float(mem_total)
    except:
        return 0.0, 0.0, 0.0


def get_gpu_temp():
    try:
        temp = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=temperature.gpu",
             "--format=csv,nounits,noheader"]
        ).decode("utf-8").strip()
        return float(temp)
    except:
        return 0.0

def batch_ocr(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    csv_log_file = os.path.join(output_dir, CSV_LOG_NAME)
    json_log_file = os.path.join(output_dir, JSON_LOG_NAME)

    # CSV header
    with open(csv_log_file, "w") as f:
        f.write("filename,status,time_sec,cpu_percent,ram_peak_mb,"
                "gpu_util,gpu_mem,gpu_total,gpu_mem_percent,gpu_temp_peak\n")

    json_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "method": "mineru"
        },
        "records": []
    }

    # files = [
    #     f for f in os.listdir(input_dir)
    #     if f.lower().endswith((".png", ".jpg", ".jpeg"))
    # ]

    files = []

    for root, dirs, filenames in os.walk(INPUT_DIR):
        for f in filenames:
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".pdf", ".tiff")):
                full_path = os.path.join(root, f)
                files.append(full_path)

    if not files:
        print("❌ No files found")
        return

    print("=" * 50)
    print("🚀 STARTING BATCH OCR (Parser Mode)")
    print("=" * 50)

    for i, filename in enumerate(files, 1):
        file_path = filename

        relative_path = os.path.relpath(file_path, input_dir)
        name_no_ext = os.path.splitext(relative_path)[0]

        output_path = os.path.join(output_dir, name_no_ext)
        os.makedirs(output_path, exist_ok=True)

        print(f"\n[{i}/{len(files)}] Processing: {filename}")

        # --- MONITOR ---
        cpu_log, ram_log, gpu_log, temp_log = [], [], [], []
        running = True

        start_time = time.time()
        current_pid = os.getpid()

        def monitor():
            while running:
                try:
                    cpu_log.append(psutil.cpu_percent(interval=None))

                    try:
                        ram = psutil.Process(current_pid).memory_info().rss / (1024 ** 2)
                        ram_log.append(ram)
                    except:
                        pass

                    util, mem_used, mem_total = get_gpu_stats()
                    gpu_log.append((util, mem_used, mem_total))

                    temp = get_gpu_temp()
                    if temp:
                        temp_log.append(temp)

                    time.sleep(1)

                except Exception as e:
                    print(f"Monitor error: {e}")
                    break

        thread = threading.Thread(target=monitor, daemon=True)
        thread.start()

        # --- RUN PARSER ---
        try:

            cmd = [
                "mineru",
                "--api-url",
                MINERU_API_URL,
                "-p",
                file_path,
                "-o",
                output_path,
                "-b",
                "vlm-engine"
            ]

            subprocess.run(
                cmd,
                capture_output=True,
                text=True,
            )

            status = "SUCCESS"

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            status = "ERROR"

        # --- STOP MONITOR ---
        running = False
        thread.join(timeout=2)

        elapsed = time.time() - start_time

        # --- METRICS ---
        cpu_avg = sum(cpu_log) / len(cpu_log) if cpu_log else 0
        ram_peak = max(ram_log) if ram_log else 0

        if gpu_log:
            gpu_util = sum(x[0] for x in gpu_log) / len(gpu_log)
            gpu_mem = sum(x[1] for x in gpu_log) / len(gpu_log)
            gpu_total = gpu_log[0][2]
            gpu_mem_percent = (gpu_mem / gpu_total * 100) if gpu_total else 0
        else:
            gpu_util = gpu_mem = gpu_total = gpu_mem_percent = 0

        temp_peak = max(temp_log) if temp_log else 0

        # --- PRINT ---
        print(f"✅ {status} | {elapsed:.2f}s")
        print(f"CPU: {cpu_avg:.1f}% | RAM peak: {ram_peak:.0f}MB")
        print(f"GPU: {gpu_util:.1f}% | VRAM: {gpu_mem:.0f}/{gpu_total:.0f}MB ({gpu_mem_percent:.1f}%)")
        if temp_peak:
            print(f"GPU Temp: {temp_peak:.1f}°C")

        # --- CSV LOG ---
        with open(csv_log_file, "a") as f:
            f.write(f"{filename},{status},{elapsed:.2f},{cpu_avg:.1f},{ram_peak:.0f},"
                    f"{gpu_util:.1f},{gpu_mem:.0f},{gpu_total:.0f},{gpu_mem_percent:.1f},{temp_peak:.1f}\n")

        # --- JSON LOG ---
        json_data["records"].append({
            "filename": filename,
            "status": status,
            "time_sec": round(elapsed, 2),
            "cpu_percent": round(cpu_avg, 1),
            "ram_peak_mb": round(ram_peak, 0),
            "gpu": {
                "util": round(gpu_util, 1),
                "mem_used": round(gpu_mem, 0),
                "mem_total": round(gpu_total, 0),
                "mem_percent": round(gpu_mem_percent, 1),
                "temp_peak": round(temp_peak, 1)
            }
        })

    # --- SAVE JSON ---
    with open(json_log_file, "w") as f:
        json.dump(json_data, f, indent=2)

    print("\n🎉 DONE")
    print(f"CSV: {csv_log_file}")
    print(f"JSON: {json_log_file}")


In [37]:
import json
import os

# Function to calculate overall summary for the processing time, resources utilization
def calculate_summary(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    records = data.get("records", [])
    if not records:
        print("No records found.")
        return

    total_files = len(records)

    # Accumulators
    total_time = 0
    total_cpu = 0
    total_ram = 0
    total_gpu_util = 0
    total_gpu_mem = 0
    total_gpu_mem_percent = 0
    total_gpu_temp = 0

    for r in records:
        total_time += r["time_sec"]
        total_cpu += r["cpu_percent"]
        total_ram += r["ram_peak_mb"]

        gpu = r.get("gpu", {})
        total_gpu_util += gpu.get("util", 0)
        total_gpu_mem += gpu.get("mem_used", 0)
        total_gpu_mem_percent += gpu.get("mem_percent", 0)
        total_gpu_temp += gpu.get("temp_peak", 0)

    # Averages
    summary = {
        "total_files": total_files,
        "avg_time_sec": total_time / total_files,
        "avg_cpu_percent": total_cpu / total_files,
        "avg_ram_peak_mb": total_ram / total_files,
        "avg_gpu_util": total_gpu_util / total_files,
        "avg_gpu_mem_mb": total_gpu_mem / total_files,
        "avg_gpu_mem_percent": total_gpu_mem_percent / total_files,
        "avg_gpu_temp": total_gpu_temp / total_files
    }

    # ---- SAVE TO TXT ----
    txt_file = os.path.splitext(json_file)[0] + "_summary.txt"

    with open(txt_file, "w") as f:
        f.write("===== OVERALL SUMMARY =====\n")
        for k, v in summary.items():
            if isinstance(v, float):
                f.write(f"{k}: {v:.2f}\n")
            else:
                f.write(f"{k}: {v}\n")

    print(f"✅ Summary saved to: {txt_file}")

    return summary


In [43]:
batch_ocr(INPUT_DIR, OUTPUT_DIR)

JSON_FILE = os.path.join(OUTPUT_DIR, JSON_LOG_NAME)
calculate_summary(JSON_FILE)

🚀 STARTING BATCH OCR (Parser Mode)

[1/20] Processing: /content/documents/level_3/table_0584.jpg
✅ SUCCESS | 20.62s
CPU: 72.5% | RAM peak: 134MB
GPU: 22.8% | VRAM: 12220/15360MB (79.6%)
GPU Temp: 78.0°C

[2/20] Processing: /content/documents/level_3/table_0804.jpg
✅ SUCCESS | 17.14s
CPU: 66.7% | RAM peak: 132MB
GPU: 18.6% | VRAM: 12294/15360MB (80.0%)
GPU Temp: 78.0°C

[3/20] Processing: /content/documents/level_3/table_0331.jpg
✅ SUCCESS | 29.40s
CPU: 70.0% | RAM peak: 132MB
GPU: 38.4% | VRAM: 12403/15360MB (80.7%)
GPU Temp: 84.0°C

[4/20] Processing: /content/documents/level_3/table_0534.jpg
✅ SUCCESS | 30.04s
CPU: 75.7% | RAM peak: 132MB
GPU: 38.6% | VRAM: 12403/15360MB (80.7%)
GPU Temp: 78.0°C

[5/20] Processing: /content/documents/level_3/table_0143.jpg
✅ SUCCESS | 21.57s
CPU: 67.1% | RAM peak: 132MB
GPU: 30.6% | VRAM: 12404/15360MB (80.8%)
GPU Temp: 77.0°C

[6/20] Processing: /content/documents/level_2/table_0281.jpg
✅ SUCCESS | 21.63s
CPU: 74.1% | RAM peak: 132MB
GPU: 17.8% | VR

{'total_files': 20,
 'avg_time_sec': 31.775,
 'avg_cpu_percent': 70.835,
 'avg_ram_peak_mb': 132.3,
 'avg_gpu_util': 24.605000000000004,
 'avg_gpu_mem_mb': 12376.15,
 'avg_gpu_mem_percent': 80.58999999999999,
 'avg_gpu_temp': 79.15}

In [44]:
import os
import re
import shutil

output_dir = "/content/output"

uuid_pattern = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"
)

deleted = 0

for item in os.listdir(output_dir):
    path = os.path.join(output_dir, item)

    if os.path.isdir(path) and uuid_pattern.match(item):
        shutil.rmtree(path, ignore_errors=True)
        deleted += 1
        print(f"Deleted: {item}")

print(f"\nRemoved {deleted} UUID folders")

Deleted: 24381945-745d-4c0c-8e8b-d7b5050db16c
Deleted: 3a70e2f1-51d8-4f34-8ff2-2b249a2ae72b
Deleted: 48d2293b-85c9-43f5-9f4e-9d415fe82285
Deleted: b3bfc2bd-a651-48d8-9b7b-ffe0a0a677db
Deleted: df44f709-ba4b-4650-8f34-a4614422c815
Deleted: 6167153d-6843-48fc-be9f-1190b16abdc1
Deleted: 8298d744-afc2-4d09-86cc-9fc8bebcfc8c
Deleted: 3f6b031c-2977-4f65-ab67-3bc04a4c7980
Deleted: 01bf6d8a-0335-465b-8098-3cefbb5830ee
Deleted: 8fe84f0d-18f1-4190-91b5-695c8206dacf
Deleted: 790c6bfa-569f-4d60-8f8f-2af3090ee569
Deleted: c4106b8b-31b1-4ac0-b5d6-ba521800f765
Deleted: 3e16020b-4445-4f08-8296-1e0bede53ef2
Deleted: 78b2b308-cf1e-4b01-8fa8-937ca574a406
Deleted: 12276194-02da-4172-920e-32ea1cb6ba70
Deleted: 4091f710-b254-4eee-ba91-39a32d4c5fbe
Deleted: d05ebfd0-1e83-4fdd-9ca4-7e5cf88caa53
Deleted: 827437e7-7bf5-4af4-b5e8-a34475b9e0d9
Deleted: e626edb4-dad2-45d2-ab1a-7d559043b2b0
Deleted: dd996775-50fd-4034-8817-4d9feb1dea5b

Removed 20 UUID folders


In [45]:
!cd /content && zip -r mineru_output.zip $OUTPUT_DIR

from google.colab import files
files.download("/content/mineru_output.zip")

  adding: output/ (stored 0%)
  adding: output/log.csv (deflated 74%)
  adding: output/level_3/ (stored 0%)
  adding: output/level_3/table_0584/ (stored 0%)
  adding: output/level_3/table_0584/table_0584/ (stored 0%)
  adding: output/level_3/table_0584/table_0584/vlm/ (stored 0%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584_origin.pdf (deflated 10%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584_layout.pdf (deflated 11%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584_content_list.json (deflated 65%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584_middle.json (deflated 88%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584_content_list_v2.json (deflated 66%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584_model.json (deflated 68%)
  adding: output/level_3/table_0584/table_0584/vlm/table_0584.md (deflated 68%)
  adding: output/level_3/table_0584/table_0584/vlm/images/ (stored 0%)
  adding: output/level

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>